# AI-Powered Telecom Customer Churn Prediction and Retention Analytics

**IBM SkillsBuild Data Analytics with AI Academic Internship Program 2026**

**Author:** Deval Shah

## Project objective
Build an end-to-end machine-learning solution that analyzes telecom customer behavior, predicts the probability that a customer will churn, ranks customers by risk, identifies important churn drivers, and converts the predictions into practical retention recommendations.

### Business problem
Telecom companies need to identify customers who are likely to leave so that retention teams can intervene before churn occurs. This project treats churn prediction as a supervised binary-classification problem and emphasizes **recall, ROC-AUC, interpretability, and actionable segmentation**, not accuracy alone.

### Dataset
IBM Telco Customer Churn sample dataset: 7,043 customer records and 21 original columns.

- IBM repository: https://github.com/IBM/telco-customer-churn-on-icp4d
- Direct CSV: https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv
- Alternative dataset page: https://www.kaggle.com/datasets/blastchar/telco-customer-churn

> **Reproducibility note:** The notebook first tries the IBM public CSV URL. If the dataset is downloaded manually, place `Telco-Customer-Churn.csv` in the same folder and the notebook will use it as a local fallback.

In [ ]:
# Core libraries
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, confusion_matrix,
                             classification_report, roc_curve, precision_recall_curve)
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

## 1. Load the dataset

In [ ]:
DATASET_URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
LOCAL_FILE = "Telco-Customer-Churn.csv"

try:
    df = pd.read_csv(DATASET_URL)
    source_used = DATASET_URL
except Exception as e:
    df = pd.read_csv(LOCAL_FILE)
    source_used = LOCAL_FILE

print("Dataset source:", source_used)
print("Shape:", df.shape)
display(df.head())

## 2. Data understanding and quality checks

In [ ]:
print("Data types:")
display(df.dtypes.to_frame("dtype"))

print("\nMissing values before cleaning:")
display(df.isna().sum().sort_values(ascending=False).to_frame("missing"))

print("\nDuplicate rows:", df.duplicated().sum())
print("\nTarget distribution:")
display(df["Churn"].value_counts().to_frame("count"))
display((df["Churn"].value_counts(normalize=True)*100).round(2).to_frame("percent"))

In [ ]:
# TotalCharges contains blank strings in the original sample; convert them to numeric.
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# Remove exact duplicate records if any and drop the identifier from modeling.
df = df.drop_duplicates().copy()

print("Rows after duplicate removal:", len(df))
print("Missing TotalCharges after conversion:", df["TotalCharges"].isna().sum())

# Basic numeric summary
display(df[["tenure", "MonthlyCharges", "TotalCharges"]].describe().T)

## 3. Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.countplot(data=df, x="Churn", ax=axes[0])
axes[0].set_title("Overall Churn Distribution")
axes[0].set_xlabel("Churn")

churn_by_contract = pd.crosstab(df["Contract"], df["Churn"], normalize="index") * 100
churn_by_contract["Yes"].sort_values().plot(kind="bar", ax=axes[1])
axes[1].set_title("Churn Rate by Contract Type")
axes[1].set_ylabel("Churn rate (%)")
axes[1].set_xlabel("Contract")
axes[1].tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.boxplot(data=df, x="Churn", y="tenure", ax=axes[0])
axes[0].set_title("Tenure by Churn")

sns.boxplot(data=df, x="Churn", y="MonthlyCharges", ax=axes[1])
axes[1].set_title("Monthly Charges by Churn")

plt.tight_layout()
plt.show()

In [ ]:
categorical_for_eda = ["InternetService", "PaymentMethod", "TechSupport"]
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, col in zip(axes, categorical_for_eda):
    rates = df.groupby(col)["Churn"].apply(lambda x: (x == "Yes").mean()*100).sort_values()
    rates.plot(kind="barh", ax=ax)
    ax.set_title(f"Churn Rate by {col}")
    ax.set_xlabel("Churn rate (%)")
plt.tight_layout()
plt.show()

## 4. Feature engineering

In [ ]:
model_df = df.copy()

# Convert target to binary.
model_df["ChurnFlag"] = (model_df["Churn"] == "Yes").astype(int)

# Useful behavior/account indicators for analytics.
model_df["ServiceCount"] = (model_df[[
    "PhoneService", "MultipleLines", "OnlineSecurity", "OnlineBackup",
    "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"
]].apply(lambda col: col.isin(["Yes", "Yes"])).sum(axis=1))

model_df["TenureGroup"] = pd.cut(
    model_df["tenure"],
    bins=[-1, 6, 12, 24, 48, 72],
    labels=["0-6 months", "7-12 months", "13-24 months", "25-48 months", "49-72 months"]
)

model_df[["tenure", "ServiceCount", "TenureGroup", "ChurnFlag"]].head()

## 5. Prepare the machine-learning dataset

In [ ]:
# Exclude identifiers and the original target text.
X = model_df.drop(columns=["customerID", "Churn", "ChurnFlag"])
y = model_df["ChurnFlag"]

numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), numeric_features),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore", drop="first"))
        ]), categorical_features)
    ]
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Training churn rate:", round(y_train.mean()*100, 2), "%")

## 6. Train and compare AI/ML models

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(
        n_estimators=300, class_weight="balanced_subsample",
        min_samples_leaf=3, random_state=RANDOM_STATE, n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=150, learning_rate=0.05, max_depth=2, random_state=RANDOM_STATE
    )
}

pipelines = {
    name: Pipeline([("preprocessor", preprocessor), ("model", model)])
    for name, model in models.items()
}

results = []
for name, pipe in pipelines.items():
    pipe.fit(X_train, y_train)
    prob = pipe.predict_proba(X_test)[:, 1]
    pred = (prob >= 0.50).astype(int)
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1": f1_score(y_test, pred, zero_division=0),
        "ROC_AUC": roc_auc_score(y_test, prob),
        "PR_AUC": average_precision_score(y_test, prob)
    })

results_df = pd.DataFrame(results).sort_values("ROC_AUC", ascending=False).reset_index(drop=True)
display(results_df.style.format({c: "{:.3f}" for c in results_df.columns if c != "Model"}))

### 6.1 Cross-validation check

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_rows = []
for name, pipe in pipelines.items():
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="roc_auc", n_jobs=-1)
    cv_rows.append({"Model": name, "CV ROC-AUC Mean": scores.mean(), "CV ROC-AUC Std": scores.std()})
cv_df = pd.DataFrame(cv_rows).sort_values("CV ROC-AUC Mean", ascending=False)
display(cv_df.style.format({"CV ROC-AUC Mean":"{:.3f}", "CV ROC-AUC Std":"{:.3f}"}))

## 7. Select the final model and evaluate it

In [ ]:
best_name = results_df.iloc[0]["Model"]
best_model = pipelines[best_name]
best_model.fit(X_train, y_train)

test_prob = best_model.predict_proba(X_test)[:, 1]
test_pred = (test_prob >= 0.50).astype(int)

print("Selected model:", best_name)
print("\nClassification report at threshold 0.50:")
print(classification_report(y_test, test_pred, target_names=["Stayed", "Churned"], zero_division=0))

cm = confusion_matrix(y_test, test_pred)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False)
plt.title(f"Confusion Matrix — {best_name}")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

## 8. Threshold analysis for retention teams

In [ ]:
threshold_rows = []
for threshold in [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]:
    pred = (test_prob >= threshold).astype(int)
    threshold_rows.append({
        "Threshold": threshold,
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1": f1_score(y_test, pred, zero_division=0),
        "Flagged_Customers": int(pred.sum())
    })
threshold_df = pd.DataFrame(threshold_rows)
display(threshold_df.style.format({"Threshold":"{:.2f}", "Precision":"{:.3f}", "Recall":"{:.3f}", "F1":"{:.3f}"}))

## 9. ROC and Precision-Recall curves

In [ ]:
fpr, tpr, _ = roc_curve(y_test, test_prob)
precision, recall, _ = precision_recall_curve(y_test, test_prob)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(fpr, tpr, label=f"AUC = {roc_auc_score(y_test, test_prob):.3f}")
axes[0].plot([0,1], [0,1], linestyle="--")
axes[0].set_title("ROC Curve")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].legend()

axes[1].plot(recall, precision, label=f"AP = {average_precision_score(y_test, test_prob):.3f}")
axes[1].set_title("Precision-Recall Curve")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].legend()

plt.tight_layout()
plt.show()

## 10. Explain important churn drivers

In [ ]:
# Permutation importance is computed on the original, human-readable input columns.
perm = permutation_importance(
    best_model, X_test, y_test, scoring="roc_auc",
    n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1
)

importance_df = pd.DataFrame({
    "Feature": X_test.columns,
    "Importance_Mean": perm.importances_mean,
    "Importance_Std": perm.importances_std
}).sort_values("Importance_Mean", ascending=False)

display(importance_df.head(12).style.format({"Importance_Mean":"{:.4f}", "Importance_Std":"{:.4f}"}))

plt.figure(figsize=(8, 5))
top = importance_df.head(10).sort_values("Importance_Mean")
plt.barh(top["Feature"], top["Importance_Mean"], xerr=top["Importance_Std"])
plt.title("Top Churn Drivers — Permutation Importance")
plt.xlabel("Decrease in ROC-AUC when feature is permuted")
plt.tight_layout()
plt.show()

## 11. Create customer risk segments

In [ ]:
scored = X_test.copy()
scored["ActualChurn"] = y_test.values
scored["ChurnProbability"] = test_prob

scored["RiskSegment"] = pd.cut(
    scored["ChurnProbability"],
    bins=[-0.001, 0.30, 0.60, 1.00],
    labels=["Low", "Medium", "High"]
)

segment_summary = scored.groupby("RiskSegment", observed=False).agg(
    Customers=("ChurnProbability", "size"),
    AverageChurnProbability=("ChurnProbability", "mean"),
    ActualChurnRate=("ActualChurn", "mean")
).reset_index()

segment_summary["ActualChurnRate"] *= 100
display(segment_summary.style.format({"AverageChurnProbability":"{:.1%}", "ActualChurnRate":"{:.1f}%"}))

## 12. AI-driven retention recommendation function

In [ ]:
def retention_recommendation(customer_row, churn_probability):
    """Convert model risk plus selected business signals into an action recommendation."""
    actions = []
    if churn_probability >= 0.60:
        priority = "URGENT"
    elif churn_probability >= 0.30:
        priority = "HIGH"
    else:
        priority = "STANDARD"

    if customer_row.get("Contract") == "Month-to-month":
        actions.append("Offer a suitable longer-term plan or contract incentive.")
    if float(customer_row.get("tenure", 0)) <= 12:
        actions.append("Trigger an early-tenure onboarding and satisfaction check.")
    if customer_row.get("TechSupport") == "No":
        actions.append("Offer a proactive technical-support or service-health check.")
    if customer_row.get("PaymentMethod") == "Electronic check":
        actions.append("Promote convenient automatic payment options if appropriate.")
    if not actions:
        actions.append("Maintain normal engagement and monitor the account.")

    return priority, actions

# Example using the first test customer.
example_customer = X_test.iloc[0].to_dict()
priority, actions = retention_recommendation(example_customer, float(test_prob[0]))
print(f"Predicted churn probability: {test_prob[0]:.1%}")
print("Priority:", priority)
for action in actions:
    print("-", action)

## 13. Final project summary

In [ ]:
final_metrics = {
    "Selected model": best_name,
    "Test ROC-AUC": roc_auc_score(y_test, test_prob),
    "Test PR-AUC": average_precision_score(y_test, test_prob),
    "Test Accuracy @ 0.50": accuracy_score(y_test, test_pred),
    "Test Precision @ 0.50": precision_score(y_test, test_pred, zero_division=0),
    "Test Recall @ 0.50": recall_score(y_test, test_pred, zero_division=0),
    "Test F1 @ 0.50": f1_score(y_test, test_pred, zero_division=0),
}
for k, v in final_metrics.items():
    print(f"{k}: {v:.3f}" if isinstance(v, float) else f"{k}: {v}")

print("\nBusiness interpretation:")
print("1. Use the model as an early-warning system, not as an automatic cancellation decision-maker.")
print("2. Prioritize high-risk customers and choose the intervention based on contract, tenure and service signals.")
print("3. Re-train and re-evaluate the model when customer behavior or product offerings change.")
print("4. Review fairness and privacy before using demographic variables in a real deployment.")

## Conclusion

This project demonstrates a complete data analytics and AI workflow: data acquisition, cleaning, exploratory analysis, feature engineering, preprocessing, supervised machine learning, model comparison, evaluation, explainability, risk segmentation, and business recommendations.

The notebook is designed so the final numerical results are generated from the actual run rather than hard-coded. This makes the submission reproducible and defensible during an internship review or viva.